In [17]:
from Secret_Key import gemini_key
import os
import streamlit as st
import pickle
import time
import langchain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

langchain.debug=True

In [18]:
os.environ['GEMINI_API_KEY'] = gemini_key

In [19]:
llm = ChatGoogleGenerativeAI(model="gemini-robotics-er-1.5-preview",temperature=0.7)

In [20]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load() 
len(data)

2

In [21]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs = r_splitter.split_documents(data)
len(docs)

18

In [22]:
docs[17]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html'}, page_content='You got 30 Day’s Trial of\n\nMoneycontrol Pro\n\nMoneycontrol Pro\n\nGrab it Now\n\nAd-Free Experience\n\nTop Articles from Financial Times\n\nTop Articles from Financial Times\n\nActionable Insights\n\nMC Research\n\nEconomic Calendar\n\n& Many More\n\n& Many More\n\nYou are already a Moneycontrol Pro user.\n\nFaizan Javed')

In [23]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorindex_geminiapi = FAISS.from_documents(docs, embeddings)
print(vectorindex_geminiapi)

In [24]:
# Storing vector index create in local
vectorindex_geminiapi.save_local("vector_index")

In [25]:
vectorIndex = FAISS.load_local(
    "vector_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [26]:
def retrieval_qa_with_sources(llm, vector_index, question, k=3):
    # Step 1: Retrieve similar documents
    docs = vector_index.similarity_search(question, k=k)

    # Step 2: Build context
    context = "\n\n".join(
        f"Source {i+1}:\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

    # Step 3: Build prompt
    prompt = f"""
You are an assistant that answers questions strictly using the provided sources.

Sources:
{context}

Question:
{question}

Answer concisely. Cite sources using [Source X].
"""

    # Step 4: Call LLM
    response = llm.invoke(prompt)

    # Step 5: Return structured output (like RetrievalQAWithSourcesChain)
    return {
        "answer": response.content,
        "sources": [f"Source {i+1}" for i in range(len(docs))],
        "source_documents": docs,
    }

In [27]:
query = "what is the price of Tiago iCNG?"

result = retrieval_qa_with_sources(
    llm=llm,
    vector_index=vectorIndex,
    question=query
)

print("Answer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

Answer:
The Tiago iCNG is priced between Rs 6.55 lakh and Rs 8.1 lakh [Source 1].

Sources:
['Source 1', 'Source 2', 'Source 3']
